In [25]:
import pandas as pd
import numpy as np
import plotly.express as px

In [26]:
df_ia = pd.read_csv("../data/raw/occ_level.csv")
df_cw_soc = pd.read_csv("../data/raw/crosswalks/soc_data.csv", dtype=str)
df_cw_cps = pd.read_csv("../data/raw/crosswalks/cps_2020_data.csv", dtype=str)
df_bls = pd.read_csv("../data/raw/bls_oews_national.csv", dtype=str)
### df_cw = pd.read_csv("../data/raw/census_crosswalk.csv", dtype=str)
df_cps = pd.read_csv("../data/raw/cps_00001.csv")


In [27]:
if "O_GROUP" in df_bls.columns:
    df_bls = df_bls[df_bls["O_GROUP"] == "detailed"].copy()

In [28]:
df_bls["TOT_EMP"] = pd.to_numeric(df_bls["TOT_EMP"].str.replace(",", ""), errors="coerce")
df_bls["A_MEDIAN"] = pd.to_numeric(df_bls["A_MEDIAN"].str.replace(",", ""), errors="coerce")
df_bls["SOC_6dig"] = df_bls["OCC_CODE"].str.strip()

In [29]:
df_bls_limpio = df_bls[["SOC_6dig", "OCC_TITLE", "TOT_EMP", "A_MEDIAN"]].dropna(subset=["SOC_6dig"])

In [30]:
df = pd.read_csv(
    '../data/raw/census_crosswalk.csv',
    usecols=[0, 1, 2, 3],
    skiprows=8,
    header=None,
    dtype=str,
)

df.columns = ['Major_Group', 'Census_Title', 'Census_Code', 'SOC_Code']

In [31]:
df = df.dropna(how='all').copy()

In [32]:
df['Major_Group'] = df['Major_Group'].str.rstrip(':').ffill()
df_clean = df[df['Census_Code'].str.match(r'^\d{4}$', na=False)].copy()
for col in df_clean.columns:
  df_clean[col] = df_clean[col].str.strip()

df_cw = df_clean.reset_index(drop=True)

df_cw

,Major_Group,Census_Title,Census_Code,SOC_Code
0,"Management, Business, Science, and Arts Occup...",Chief executives,0010,11-1011
1,"Management, Business, Science, and Arts Occup...",General and operations managers,0020,11-1021
2,"Management, Business, Science, and Arts Occup...",Legislators,0030,11-1031
3,"Management, Business, Science, and Arts Occup...",Advertising and promotions managers,0040,11-2011
4,"Management, Business, Science, and Arts Occup...",Marketing managers,0051,11-2021
...,...,...,...,...
565,Military Specific Occupations,Military officer special and tactical operatio...,9800,55-1010
566,Military Specific Occupations,First-line enlisted military supervisors,9810,55-2010
567,Military Specific Occupations,Military enlisted tactical operations and air/...,9825,55-3010
568,Military Specific Occupations,"Military, rank not Specified",9830,none


In [33]:
df_cps['OCC_clean'] = (
    df_cps['OCC'].fillna(0).astype(int).astype(str).str.zfill(4)
)
df_cw['Census_Code'] = df_cw['Census_Code'].astype(str).str.zfill(4)

In [34]:
df_merged = pd.merge(
    df_cps,
    df_cw,
    left_on='OCC_clean',
    right_on='Census_Code',
    how='left',
)

df_merged

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,EMPSTAT,OCC,OCC2010,DURUNEMP,EDUC,OCC_clean,Major_Group,Census_Title,Census_Code,SOC_Code
0,2021,1,1,1805.3669,20191100000100,NaN,NaN,1,1805.3669,20191100000101,...,32,0,9999,999,81,0000,NaN,NaN,NaN,NaN
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,10,4150,4150,999,73,4150,Service Occupations,"Hosts and hostesses, restaurant, lounge, and c...",4150,35-9031
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,21,4110,4110,39,111,4110,Service Occupations,Waiters and waitresses,4110,35-3031
3,2021,2,1,1903.8219,20201100000100,NaN,NaN,1,1903.8219,20201100000101,...,36,0,9999,999,123,0000,NaN,NaN,NaN,NaN
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,10,3602,4610,999,73,3602,Service Occupations,Personal care aides,3602,31-1122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,10,850,850,999,111,0850,"Management, Business, Science, and Arts Occup...",Personal financial advisors,0850,13-2052
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,10,3930,3930,999,123,3930,Service Occupations,Security guards and gambling surveillance offi...,3930,33-9030
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,10,3603,3600,999,92,3603,Service Occupations,Nursing assistants,3603,31-1131
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,10,2014,2010,999,123,2014,"Management, Business, Science, and Arts Occup...","Social workers, all other",2014,21-1029


In [35]:
df_merged = df_merged.drop(columns=['OCC_clean', 'Census_Code'])

In [36]:
df_merged['SOC_Code'] = df_merged['SOC_Code'].astype(str).str.strip()
df_bls_limpio['SOC_6dig'] = df_bls_limpio['SOC_6dig'].astype(str).str.strip()

In [37]:
df_final = pd.merge(
    df_merged,
    df_bls_limpio,
    left_on='SOC_Code',
    right_on='SOC_6dig',
    how='left',
)

In [38]:
df_bls_limpio['SOC_prefijo'] = df_bls_limpio['SOC_6dig'].str[:6]
bls_grupo = df_bls_limpio.groupby('SOC_prefijo', as_index=False).agg({
    'TOT_EMP': 'sum',
    'A_MEDIAN': 'mean',
})

In [39]:
df_final['SOC_prefijo'] = df_final['SOC_Code'].str[:6]
df_final = pd.merge(
    df_final, bls_grupo, on='SOC_prefijo', how='left', suffixes=('', '_grupo')
)

In [40]:
df_final['TOT_EMP'] = df_final['TOT_EMP'].fillna(df_final['TOT_EMP_grupo'])
df_final['A_MEDIAN'] = df_final['A_MEDIAN'].fillna(df_final['A_MEDIAN_grupo'])

In [41]:
df_final = df_final.drop(
    columns=['SOC_6dig', 'SOC_prefijo', 'TOT_EMP_grupo', 'A_MEDIAN_grupo']
)

In [42]:
df_final

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,OCC,OCC2010,DURUNEMP,EDUC,Major_Group,Census_Title,SOC_Code,OCC_TITLE,TOT_EMP,A_MEDIAN
0,2021,1,1,1805.3669,20191100000100,NaN,NaN,1,1805.3669,20191100000101,...,0,9999,999,81,NaN,NaN,NaN,NaN,NaN,NaN
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,4150,4150,999,73,Service Occupations,"Hosts and hostesses, restaurant, lounge, and c...",35-9031,"Hosts and Hostesses, Restaurant, Lounge, and C...",432690.0,31200.0
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,4110,4110,39,111,Service Occupations,Waiters and waitresses,35-3031,Waiters and Waitresses,2270910.0,35230.0
3,2021,2,1,1903.8219,20201100000100,NaN,NaN,1,1903.8219,20201100000101,...,0,9999,999,123,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,3602,4610,999,73,Service Occupations,Personal care aides,31-1122,NaN,4305810.0,35800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,850,850,999,111,"Management, Business, Science, and Arts Occup...",Personal financial advisors,13-2052,Personal Financial Advisors,266800.0,105070.0
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,3930,3930,999,123,Service Occupations,Security guards and gambling surveillance offi...,33-9030,NaN,1292990.0,40695.0
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,3603,3600,999,92,Service Occupations,Nursing assistants,31-1131,Nursing Assistants,1448910.0,42260.0
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,2014,2010,999,123,"Management, Business, Science, and Arts Occup...","Social workers, all other",21-1029,"Social Workers, All Other",62930.0,71900.0


In [43]:
col_soc_ia = "O*NET-SOC Code"

df_ia[col_soc_ia] = df_ia[col_soc_ia].astype(str).str.strip()


In [44]:
df_completo = pd.merge(
    df_final, df_ia, left_on='SOC_Code', right_on=col_soc_ia, how='left'
)

df_completo

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,TOT_EMP,A_MEDIAN,O*NET-SOC Code,Title,dv_rating_alpha,dv_rating_beta,dv_rating_gamma,human_rating_alpha,human_rating_beta,human_rating_gamma
0,2021,1,1,1805.3669,20191100000100,NaN,NaN,1,1805.3669,20191100000101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,432690.0,31200.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,2270910.0,35230.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021,2,1,1903.8219,20201100000100,NaN,NaN,1,1903.8219,20201100000101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,4305810.0,35800.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,266800.0,105070.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,1292990.0,40695.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,1448910.0,42260.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,62930.0,71900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
cols_numericas_ia = df_ia.select_dtypes(include='number').columns.tolist()
df_ia['SOC_prefijo'] = df_ia[col_soc_ia].str[:6]
ia_grupo = df_ia.groupby('SOC_prefijo', as_index=False)[cols_numericas_ia].mean()

df_completo['SOC_prefijo'] = df_completo['SOC_Code'].str[:6]
df_completo = pd.merge(
    df_completo, ia_grupo, on='SOC_prefijo', how='left', suffixes=('', '_grupo')
)

for col in cols_numericas_ia:
  df_completo[col] = df_completo[col].fillna(df_completo[f'{col}_grupo'])

df_completo

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,human_rating_alpha,human_rating_beta,human_rating_gamma,SOC_prefijo,dv_rating_alpha_grupo,dv_rating_beta_grupo,dv_rating_gamma_grupo,human_rating_alpha_grupo,human_rating_beta_grupo,human_rating_gamma_grupo
0,2021,1,1,1805.3669,20191100000100,NaN,NaN,1,1805.3669,20191100000101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,0.200000,0.328571,0.457143,35-903,0.057143,0.228571,0.400000,0.200000,0.328571,0.457143
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,0.204082,0.224490,0.244898,35-303,0.081633,0.183673,0.285714,0.204082,0.224490,0.244898
3,2021,2,1,1903.8219,20201100000100,NaN,NaN,1,1903.8219,20201100000101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,0.111111,0.185897,0.260684,31-112,0.121795,0.149573,0.177350,0.111111,0.185897,0.260684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,0.231193,0.563596,0.896000,13-205,0.065539,0.516575,0.967611,0.231193,0.563596,0.896000
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,0.083333,0.281250,0.479167,33-903,0.208333,0.302083,0.395833,0.083333,0.281250,0.479167
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,0.011299,0.102489,0.193679,31-113,0.058616,0.075565,0.092514,0.011299,0.102489,0.193679
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,0.175198,0.358681,0.542163,21-102,0.029464,0.285813,0.542163,0.175198,0.358681,0.542163


In [46]:
cols_borrar = [col_soc_ia, 'SOC_prefijo'] + [
    f'{c}_grupo' for c in cols_numericas_ia
]
df_completo = df_completo.drop(
    columns=[c for c in cols_borrar if c in df_completo.columns]
)

In [47]:
ocupados = df_completo[df_completo['OCC'] != 0]

ocupados

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,OCC_TITLE,TOT_EMP,A_MEDIAN,Title,dv_rating_alpha,dv_rating_beta,dv_rating_gamma,human_rating_alpha,human_rating_beta,human_rating_gamma
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,"Hosts and Hostesses, Restaurant, Lounge, and C...",432690.0,31200.0,NaN,0.057143,0.228571,0.400000,0.200000,0.328571,0.457143
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,Waiters and Waitresses,2270910.0,35230.0,NaN,0.081633,0.183673,0.285714,0.204082,0.224490,0.244898
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,NaN,4305810.0,35800.0,NaN,0.121795,0.149573,0.177350,0.111111,0.185897,0.260684
5,2021,3,1,2738.9084,20191200000300,NaN,NaN,2,2738.9084,20191200000302,...,Receptionists and Information Clerks,910180.0,38010.0,NaN,0.200000,0.533333,0.866667,0.433333,0.583333,0.733333
8,2021,4,1,1709.2628,20210100000400,NaN,NaN,2,2036.5968,20210100000402,...,Cardiovascular Technologists and Technicians,62960.0,74310.0,NaN,0.074990,0.349919,0.624848,0.124292,0.253648,0.383005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,Personal Financial Advisors,266800.0,105070.0,NaN,0.065539,0.516575,0.967611,0.231193,0.563596,0.896000
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,NaN,1292990.0,40695.0,NaN,0.208333,0.302083,0.395833,0.083333,0.281250,0.479167
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,Nursing Assistants,1448910.0,42260.0,NaN,0.058616,0.075565,0.092514,0.011299,0.102489,0.193679
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,"Social Workers, All Other",62930.0,71900.0,NaN,0.029464,0.285813,0.542163,0.175198,0.358681,0.542163
